<a href="https://colab.research.google.com/github/it0770e/xai-ids/blob/main/Notebook03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1
from google.colab import drive
drive.mount('/content/drive')
!pip install imbalanced-learn -q
import pandas as pd, numpy as np, joblib, warnings, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
warnings.filterwarnings('ignore')
BASE = '/content/drive/MyDrive/xai-ids'
print('✅ Setup complete')

Mounted at /content/drive
✅ Setup complete


In [ ]:
# CELL 2
df_train = pd.read_pickle(f'{BASE}/data/raw/nsl_kdd_train.pkl')
df_test  = pd.read_pickle(f'{BASE}/data/raw/nsl_kdd_test.pkl')
print(f'✅ Train: {df_train.shape}  |  Test: {df_test.shape}')

✅ Train: (125973, 45)  |  Test: (22544, 45)


In [ ]:
# CELL 3
drop_cols = ['label','binary_label','attack_category','difficulty']
y_train_raw = df_train['binary_label'].values
y_test      = df_test['binary_label'].values
X_train_raw = df_train.drop(columns=drop_cols)
X_test_raw  = df_test.drop(columns=drop_cols)
feature_names_ordered = X_train_raw.columns.tolist()
print(f'✅ Feature matrix: {X_train_raw.shape}')
print(f'   Feature names saved: {len(feature_names_ordered)}')
print(f'   First 5: {feature_names_ordered[:5]}')

✅ Feature matrix: (125973, 41)
   Feature names saved: 41
   First 5: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes']


In [ ]:
# CELL 4
cat_cols = X_train_raw.select_dtypes(include='object').columns.tolist()
num_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
print(f'Categorical: {cat_cols}')
print(f'Numerical  : {len(num_cols)} columns')

label_encoders = {}
X_train_enc = X_train_raw.copy()
X_test_enc  = X_test_raw.copy()

for col in cat_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train_raw[col])
    X_test_enc[col]  = X_test_raw[col].map(
        lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
    )
    label_encoders[col] = le
    print(f'   {col}: {len(le.classes_)} categories')
print('✅ Encoding done')

Categorical: ['protocol_type', 'service', 'flag']
Numerical  : 38 columns
   protocol_type: 3 categories
   service: 70 categories
   flag: 11 categories
✅ Encoding done


In [ ]:
# CELL 5
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_enc)
X_test_scaled  = scaler.transform(X_test_enc)
print('✅ StandardScaler applied (fit on train only — no data leakage)')
print(f'   Train mean (first 3): {X_train_scaled.mean(axis=0)[:3].round(4)}')
print(f'   Train std  (first 3): {X_train_scaled.std(axis=0)[:3].round(4)}')

✅ StandardScaler applied (fit on train only — no data leakage)
   Train mean (first 3): [ 0. -0.  0.]
   Train std  (first 3): [1. 1. 1.]


In [ ]:
# CELL 6
X_train, X_val, y_train, y_val = train_test_split(
    X_train_scaled, y_train_raw,
    test_size=0.18, random_state=42, stratify=y_train_raw
)
print('✅ Stratified split:')
print(f'   Train: {X_train.shape}  classes: {np.bincount(y_train)}')
print(f'   Val  : {X_val.shape}   classes: {np.bincount(y_val)}')
print(f'   Test : {X_test_scaled.shape}  classes: {np.bincount(y_test)}')

✅ Stratified split:
   Train: (103297, 41)  classes: [55221 48076]
   Val  : (22676, 41)   classes: [12122 10554]
   Test : (22544, 41)  classes: [ 9711 12833]


In [ ]:
# CELL 7
print('Before SMOTE:')
for cls, cnt in zip(*np.unique(y_train, return_counts=True)):
    print(f'   Class {cls}: {cnt:,} ({cnt/len(y_train):.1%})')

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('After SMOTE:')
for cls, cnt in zip(*np.unique(y_train_bal, return_counts=True)):
    print(f'   Class {cls}: {cnt:,} ({cnt/len(y_train_bal):.1%})')

fig, axes = plt.subplots(1, 2, figsize=(10,4))
for ax, y, title in zip(axes, [y_train, y_train_bal], ['Before SMOTE','After SMOTE']):
    counts = np.bincount(y)
    ax.bar(['Normal','Attack'], counts, color=['#2ecc71','#e74c3c'])
    ax.set_title(title); ax.set_ylabel('Samples')
    for i,c in enumerate(counts): ax.text(i, c+50, f'{c:,}', ha='center')
plt.suptitle('Class Balance Before and After SMOTE')
plt.tight_layout()
plt.savefig(f'{BASE}/results/figures/smote_balance.png', dpi=300, bbox_inches='tight')
plt.show()

Before SMOTE:
   Class 0: 55,221 (53.5%)
   Class 1: 48,076 (46.5%)


In [ ]:
# CELL 8
processed_data = {
    'X_train': X_train_bal, 'X_val': X_val, 'X_test': X_test_scaled,
    'y_train': y_train_bal, 'y_val': y_val, 'y_test': y_test,
    'feature_names': feature_names_ordered
}
joblib.dump(processed_data, f'{BASE}/data/processed/nsl_kdd_processed.pkl')

preprocessors = {
    'label_encoders': label_encoders, 'scaler': scaler,
    'categorical_cols': cat_cols, 'numerical_cols': num_cols,
    'feature_names': feature_names_ordered
}
joblib.dump(preprocessors, f'{BASE}/results/models/preprocessors.pkl')
print('✅ Processed data saved (feature_names included)')
print('✅ Preprocessors saved')
print('🎉 Ready for Notebook 04')